# Phase 1: Behavioral Sweep — PI > RI in Transformers

Full interference landscape mapping for Qwen2.5-0.5B-Instruct.

- Grid: 12 key levels x 17 update levels (capped at 200) x 2 conditions (RI/PI)
- 30 trials per cell with bootstrap 95% CIs
- Early stopping: 3 consecutive zero-accuracy cells for EITHER condition → skip rest of row
- Resume from partial saves

In [1]:
# Install dependencies
!pip install -q transformers accelerate scipy

In [35]:
import os
from pathlib import Path

# Colab runs at /content — use this for saves (persists within session)
SAVE_DIR = '/content/results'
FIGURES_DIR = '/content/figures'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Google Drive backup — survives runtime disconnects
SAVE_DIR_DRIVE = '/content/drive/MyDrive/mechanistic_probing_results'
os.makedirs(SAVE_DIR_DRIVE, exist_ok=True)

print(f'Results: {SAVE_DIR}')
print(f'Drive backup: {SAVE_DIR_DRIVE}')
print(f'Figures: {FIGURES_DIR}')
print(f'Existing results: {os.listdir(SAVE_DIR)}')

Results: /content/results
Drive backup: /content/drive/MyDrive/mechanistic_probing_results
Figures: /content/figures
Existing results: ['behavioral_sweep_Qwen2.5-0.5B-Instruct_full_partial.json', 'behavioral_sweep_Qwen2.5-0.5B-Instruct_partial.json']


In [36]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Copy results to Drive
# import shutil
# shutil.copy(
#     '/content/results/behavioral_sweep_Qwen2.5-0.5B-Instruct_partial.json',
#     '/content/drive/MyDrive/behavioral_sweep_partial.json'
# )
# shutil.copy(
#     '/content/results/behavioral_sweep_Qwen2.5-0.5B-Instruct_full_partial.json',
#     '/content/drive/MyDrive/behavioral_sweep_full_partial.json'
# )
# print("Copied to Google Drive — download from drive.google.com")

In [37]:
import torch
import json
import time
import random
import numpy as np
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


## Dataset Generation Module

In [38]:
# ═══════════════════════════════════════════════════════════════════════════
# Dataset generation — self-contained, no external imports
# ═══════════════════════════════════════════════════════════════════════════

ORIGINAL_CATEGORIES = [
    "visual art", "tools", "landform", "musical instrument", "gemstone",
    "fabric", "tree species", "cheese variety", "architectural style",
    "cloud formation", "bird species", "culinary herb", "flower species",
    "wine variety", "dance style", "pasta shape", "literary genre",
    "cooking method", "mathematical concept", "weather phenomenon",
    "ocean current", "mineral type", "coffee variety", "telescope type",
    "martial art", "sea creature", "psychology term", "chemical element",
    "dinosaur genus", "programming language", "ancient civilization",
    "bridge type", "photography technique", "boat type", "cartoon character",
    "stadium name", "surgical procedure", "constellation", "spice blend",
    "guitar type", "hat style", "painting medium", "volcano name",
    "fruit variety", "sword type", "board game",
]

PREFIX_MAP = {
    "visual art": "Art", "tools": "Tool", "landform": "Land",
    "musical instrument": "Inst", "gemstone": "Gem", "fabric": "Fab",
    "tree species": "Tree", "cheese variety": "Cheese",
    "architectural style": "Style", "cloud formation": "Cloud",
    "bird species": "Bird", "culinary herb": "Herb",
    "flower species": "Flower", "wine variety": "Wine",
    "dance style": "Dance", "pasta shape": "Pasta",
    "literary genre": "Genre", "cooking method": "Cook",
    "mathematical concept": "Math", "weather phenomenon": "Weather",
    "ocean current": "Current", "mineral type": "Mineral",
    "coffee variety": "Coffee", "telescope type": "Scope",
    "martial art": "Martial", "sea creature": "Sea",
    "psychology term": "Psych", "chemical element": "Elem",
    "dinosaur genus": "Dino", "programming language": "Lang",
    "ancient civilization": "Civ", "bridge type": "Bridge",
    "photography technique": "Photo", "boat type": "Boat",
    "cartoon character": "Toon", "stadium name": "Stadium",
    "surgical procedure": "Surg", "constellation": "Star",
    "spice blend": "Spice", "guitar type": "Guitar",
    "hat style": "Hat", "painting medium": "Medium",
    "volcano name": "Volcano", "fruit variety": "Fruit",
    "sword type": "Sword", "board game": "Game",
}

KEY_LEVELS = [2, 3, 5, 7, 10, 15, 20, 25, 30, 35, 40, 46]
# Capped at 200 — pattern is clear by then, higher just wastes GPU time
UPDATE_LEVELS = [
    1, 3, 5, 10, 15, 20,
    30, 40, 50, 60,
    80, 100, 120, 140, 160, 180, 200,
]

SYSTEM_PROMPT = "Answer with ONLY the exact value. No explanation."


def generate_value_pool(category, pool_size=500):
    prefix = PREFIX_MAP.get(category, category.split()[0].capitalize())
    return [f"{prefix}{i}" for i in range(1, pool_size + 1)]


def shuffle_no_consecutive(items, rng, max_attempts=100):
    """Shuffle ensuring no consecutive same-category items."""
    for _ in range(max_attempts):
        candidate = items.copy()
        rng.shuffle(candidate)
        ok = True
        for i in range(1, len(candidate)):
            if candidate[i]["category"] == candidate[i-1]["category"]:
                ok = False
                break
        if ok:
            return candidate
    # Fallback: greedy
    remaining = items.copy()
    rng.shuffle(remaining)
    result = []
    last_cat = None
    while remaining:
        valid = [i for i, item in enumerate(remaining) if item["category"] != last_cat]
        if not valid:
            result.extend(remaining)
            break
        idx = rng.choice(valid)
        item = remaining.pop(idx)
        result.append(item)
        last_cat = item["category"]
    return result


def generate_trial(num_keys, num_updates, condition, seed, test_category_idx=0):
    rng = random.Random(seed)
    categories = rng.sample(ORIGINAL_CATEGORIES, min(num_keys, len(ORIGINAL_CATEGORIES)))
    
    values_per_cat = {}
    for cat in categories:
        pool = generate_value_pool(cat, pool_size=max(num_updates + 50, 500))
        selected = rng.sample(pool, num_updates)
        values_per_cat[cat] = selected
    
    test_category = categories[test_category_idx % len(categories)]
    
    # Build interleaved sequence
    items = []
    for cat in categories:
        for idx, val in enumerate(values_per_cat[cat]):
            items.append({"category": cat, "value": val, "update_idx": idx})
    sequence = shuffle_no_consecutive(items, rng)
    
    # Build prompt
    stream_lines = [f"{item['category']}: {item['value']}" for item in sequence]
    stream_text = "\n".join(stream_lines)
    query_word = "first" if condition == "RI" else "last"
    cat_values = [item["value"] for item in sequence if item["category"] == test_category]
    expected = cat_values[0] if condition == "RI" else cat_values[-1]
    
    prompt = (
        f"Read the following key-value stream. Each key gets updated multiple times.\n\n"
        f"{stream_text}\n\n"
        f"What was the {query_word} value of {test_category}?"
    )
    
    return {
        "prompt": prompt,
        "expected": expected,
        "condition": condition,
        "test_category": test_category,
        "initial_value": cat_values[0],
        "final_value": cat_values[-1],
        "all_values": cat_values,
        "num_keys": num_keys,
        "num_updates": num_updates,
        "seed": seed,
    }


def format_for_chat(prompt, tokenizer):
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


print(f"Categories: {len(ORIGINAL_CATEGORIES)}")
print(f"Grid: {len(KEY_LEVELS)} key levels x {len(UPDATE_LEVELS)} update levels = {len(KEY_LEVELS)*len(UPDATE_LEVELS)} cells")

Categories: 46
Grid: 12 key levels x 17 update levels = 204 cells


## Load Model

In [39]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map=DEVICE,
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_params = sum(p.numel() for p in model.parameters())
ctx_limit = model.config.max_position_embeddings
print(f"Loaded: {n_params:,} params, ctx={ctx_limit}")

Loading Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: 494,032,768 params, ctx=32768


## Pre-flight Context Check

In [40]:
def preflight_context_check(num_keys, num_updates, tokenizer, context_limit, safety_margin=0.90):
    trial = generate_trial(num_keys, num_updates, "RI", seed=0)
    formatted = format_for_chat(trial["prompt"], tokenizer)
    tokens = tokenizer.encode(formatted)
    n_tokens = len(tokens)
    limit = int(context_limit * safety_margin)
    return n_tokens <= limit, n_tokens


# Compute feasible grid
feasible = {}
for nk in KEY_LEVELS:
    for nu in UPDATE_LEVELS:
        ok, n_tokens = preflight_context_check(nk, nu, tokenizer, ctx_limit)
        if ok:
            feasible[(nk, nu)] = n_tokens
        else:
            break  # higher updates won't fit either

print(f"Feasible cells: {len(feasible)} / {len(KEY_LEVELS) * len(UPDATE_LEVELS)}")
print(f"\n{'Keys':>6} | Max updates | Est. tokens")
print(f"{'-'*6}-+-{'-'*11}-+-{'-'*11}")
for nk in KEY_LEVELS:
    max_nu = max((nu for (k, nu) in feasible if k == nk), default=0)
    if max_nu > 0:
        est = feasible[(nk, max_nu)]
        print(f"{nk:>6} | {max_nu:>11} | {est:>11,}")
    else:
        print(f"{nk:>6} | {'NONE':>11} | {'N/A':>11}")

Feasible cells: 175 / 204

  Keys | Max updates | Est. tokens
-------+-------------+------------
     2 |         200 |       3,556
     3 |         200 |       5,113
     5 |         200 |       8,614
     7 |         200 |      11,730
    10 |         200 |      16,588
    15 |         200 |      25,377
    20 |         160 |      27,007
    25 |         120 |      25,399
    30 |         100 |      25,263
    35 |         100 |      29,456
    40 |          80 |      26,912
    46 |          60 |      23,049


## Sweep Engine

In [46]:
def classify_error(predicted, expected, initial_value, final_value, all_values, condition):
    pred_lower = predicted.lower().strip()
    exp_lower = expected.lower().strip()
    if exp_lower in pred_lower or pred_lower.startswith(exp_lower):
        return "correct"
    init_lower = initial_value.lower()
    final_lower = final_value.lower()
    if condition == "PI" and (init_lower in pred_lower or pred_lower.startswith(init_lower)):
        return "primacy_intrusion"
    if condition == "RI" and (final_lower in pred_lower or pred_lower.startswith(final_lower)):
        return "recency_intrusion"
    for val in all_values:
        if val.lower() != exp_lower and (val.lower() in pred_lower or pred_lower.startswith(val.lower())):
            return "intermediate_intrusion"
    return "garbage"


def bootstrap_ci(data, n_bootstrap=2000, ci=0.95):
    if not data:
        return 0.0, 0.0, 0.0
    arr = np.array(data, dtype=float)
    mean = arr.mean()
    if len(arr) < 3:
        return mean, 0.0, 1.0
    rng = np.random.RandomState(42)
    boot_means = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_bootstrap)]
    alpha = (1 - ci) / 2
    return mean, np.percentile(boot_means, alpha * 100), np.percentile(boot_means, (1 - alpha) * 100)


# Token threshold: clear cache between individual trials when prompts are this long
CACHE_CLEAR_TOKEN_THRESHOLD = 8000


def run_single_trial(model, tokenizer, trial, max_new_tokens=20):
    formatted = format_for_chat(trial["prompt"], tokenizer)
    inputs = tokenizer(formatted, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    n_tokens = inputs["input_ids"].shape[1]

    # For long sequences, clear cache before running to maximize free memory
    if n_tokens > CACHE_CLEAR_TOKEN_THRESHOLD:
        torch.cuda.empty_cache()

    try:
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=None, top_p=None,
            )
        new_ids = gen_ids[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip().split("\n")[0].strip()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        answer = "[OOM]"

    # Free the input/output tensors
    del inputs
    if n_tokens > CACHE_CLEAR_TOKEN_THRESHOLD:
        torch.cuda.empty_cache()
    
    error_type = classify_error(
        answer, trial["expected"],
        trial["initial_value"], trial["final_value"],
        trial["all_values"], trial["condition"],
    )
    return {
        "seed": trial["seed"],
        "num_keys": trial["num_keys"],
        "num_updates": trial["num_updates"],
        "condition": trial["condition"],
        "test_category": trial["test_category"],
        "expected": trial["expected"],
        "predicted": answer,
        "correct": error_type == "correct",
        "error_type": error_type,
        "initial_value": trial["initial_value"],
        "final_value": trial["final_value"],
    }


print("Sweep engine ready.")

Sweep engine ready.


In [42]:
# Quick sanity test: keys=3, updates=3, 5 trials
print("--- Sanity check ---")
for seed in range(5):
    for cond in ["RI", "PI"]:
        trial = generate_trial(3, 3, cond, seed=seed+100)
        result = run_single_trial(model, tokenizer, trial)
        status = "OK" if result["correct"] else f"WRONG ({result['error_type']})"
        print(f"  seed={seed} {cond}: exp={result['expected']:>12} got={result['predicted'][:15]:>15} {status}")

--- Sanity check ---
  seed=0 RI: exp=    Cloud375 got=       Cloud375 OK
  seed=0 PI: exp=    Cloud362 got=       Cloud362 OK
  seed=1 RI: exp=     Star240 got=        Star240 OK
  seed=1 PI: exp=     Star474 got=        Star474 OK
  seed=2 RI: exp=    Cloud284 got=       Cloud284 OK
  seed=2 PI: exp=    Cloud184 got=       Cloud184 OK
  seed=3 RI: exp=     Game111 got=        Game111 OK
  seed=3 PI: exp=     Game377 got=        Game411 WRONG (garbage)
  seed=4 RI: exp=       Tool4 got=        Tool170 WRONG (intermediate_intrusion)
  seed=4 PI: exp=     Tool201 got=          Tool4 WRONG (primacy_intrusion)


## Run Full Sweep

In [47]:
TRIALS_PER_CELL = 30

def save_results(results, partial=False):
    """Save summary and full results to local + Google Drive."""
    import shutil
    model_short = MODEL_NAME.split("/")[-1]
    suffix = "_partial" if partial else ""
    
    # Summary (without individual trials)
    summary = {k: v for k, v in results.items() if k != "cells"}
    summary["cells"] = {}
    for cell_key, cell_data in results["cells"].items():
        summary["cells"][cell_key] = {k: v for k, v in cell_data.items() if k != "trials"}
    
    summary_path = f"{SAVE_DIR}/behavioral_sweep_{model_short}{suffix}.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    
    # Full data with trials
    full_path = f"{SAVE_DIR}/behavioral_sweep_{model_short}_full{suffix}.json"
    with open(full_path, "w") as f:
        json.dump(results, f, indent=2)
    
    # Backup to Google Drive
    try:
        shutil.copy(summary_path, f"{SAVE_DIR_DRIVE}/behavioral_sweep_{model_short}{suffix}.json")
        shutil.copy(full_path, f"{SAVE_DIR_DRIVE}/behavioral_sweep_{model_short}_full{suffix}.json")
        print(f"  -> Saved to {summary_path} + Drive backup")
    except Exception as e:
        print(f"  -> Saved to {summary_path} (Drive backup failed: {e})")


# Initialize or resume — check Drive first, then local
model_short = MODEL_NAME.split('/')[-1]
resume_path = f"{SAVE_DIR}/behavioral_sweep_{model_short}_full_partial.json"
drive_resume_path = f"{SAVE_DIR_DRIVE}/behavioral_sweep_{model_short}_full_partial.json"

if os.path.exists(resume_path):
    print(f"Resuming from {resume_path}")
    with open(resume_path) as f:
        results = json.load(f)
    print(f"  {len(results['cells'])} cells already completed")
elif os.path.exists(drive_resume_path):
    print(f"Resuming from Drive backup: {drive_resume_path}")
    with open(drive_resume_path) as f:
        results = json.load(f)
    print(f"  {len(results['cells'])} cells already completed")
else:
    results = {
        "model": MODEL_NAME,
        "config": {
            "key_levels": KEY_LEVELS,
            "update_levels": UPDATE_LEVELS,
            "trials_per_cell": TRIALS_PER_CELL,
            "num_categories": len(ORIGINAL_CATEGORIES),
        },
        "feasible_cells": {f"{k}_{u}": t for (k, u), t in feasible.items()},
        "cells": {},
        "start_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

total_cells = len(feasible)
completed = len(results["cells"])
print(f"\nSweep: {total_cells} feasible cells, {TRIALS_PER_CELL} trials/cell, "
      f"2 conditions = {total_cells * TRIALS_PER_CELL * 2} total trials")
print(f"Already completed: {completed}")

Resuming from /content/results/behavioral_sweep_Qwen2.5-0.5B-Instruct_full_partial.json
  70 cells already completed

Sweep: 175 feasible cells, 30 trials/cell, 2 conditions = 10500 total trials
Already completed: 70


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MAIN SWEEP LOOP
# - Early stopping: 3 consecutive 0% for EITHER RI or PI → skip rest of row
# - Resumes from partial saves automatically
# - torch.cuda.empty_cache() after each cell to prevent OOM on long sequences
# ═══════════════════════════════════════════════════════════════════════════

cell_idx = 0
saturation_tracker = {}  # nk -> {"RI": consecutive_zeros, "PI": consecutive_zeros}

sweep_start = time.time()

for nk in KEY_LEVELS:
    saturation_tracker.setdefault(nk, {"RI": 0, "PI": 0})

    for nu in UPDATE_LEVELS:
        if (nk, nu) not in feasible:
            continue

        cell_key = f"{nk}_{nu}"

        # Skip already completed
        if cell_key in results["cells"]:
            cell_idx += 1
            # Rebuild saturation tracker from existing data
            for cond in ["RI", "PI"]:
                acc = results["cells"][cell_key]["stats"][cond]["accuracy"]
                if acc == 0.0:
                    saturation_tracker[nk][cond] += 1
                else:
                    saturation_tracker[nk][cond] = 0
            continue

        # Early stopping: OR condition, 3 consecutive zeros
        if saturation_tracker[nk]["RI"] >= 3 or saturation_tracker[nk]["PI"] >= 3:
            print(f"  [{cell_idx+1}/{total_cells}] keys={nk}, updates={nu}: SKIPPED (saturated - "
                  f"RI zeros={saturation_tracker[nk]['RI']}, PI zeros={saturation_tracker[nk]['PI']})")
            cell_idx += 1
            continue

        cell_start = time.time()
        cell_results = {"RI": [], "PI": []}

        for condition in ["RI", "PI"]:
            for trial_idx in range(TRIALS_PER_CELL):
                seed = hash((nk, nu, condition, trial_idx)) % (2**31)
                test_cat_idx = trial_idx % min(nk, len(ORIGINAL_CATEGORIES))
                trial = generate_trial(nk, nu, condition, seed=seed, test_category_idx=test_cat_idx)
                result = run_single_trial(model, tokenizer, trial)
                cell_results[condition].append(result)

        # Free GPU memory from KV cache (model weights stay loaded)
        torch.cuda.empty_cache()

        # Compute cell statistics
        cell_stats = {}
        for cond in ["RI", "PI"]:
            corrects = [r["correct"] for r in cell_results[cond]]
            mean, ci_lo, ci_hi = bootstrap_ci(corrects)
            error_counts = {}
            for r in cell_results[cond]:
                et = r["error_type"]
                error_counts[et] = error_counts.get(et, 0) + 1
            cell_stats[cond] = {
                "accuracy": mean, "ci_lower": ci_lo, "ci_upper": ci_hi,
                "n": len(corrects), "error_types": error_counts,
            }
            if mean == 0.0:
                saturation_tracker[nk][cond] += 1
            else:
                saturation_tracker[nk][cond] = 0

        # Classify regime
        ri_acc = cell_stats["RI"]["accuracy"]
        pi_acc = cell_stats["PI"]["accuracy"]
        if ri_acc >= 0.5 and pi_acc >= 0.5:
            regime = "A"
        elif ri_acc >= 0.5 and pi_acc < 0.5:
            regime = "B"
        elif ri_acc < 0.5 and pi_acc < 0.5:
            regime = "C"
        else:
            regime = "D"  # PI > RI (unexpected)

        cell_data = {
            "num_keys": nk, "num_updates": nu,
            "stats": cell_stats, "regime": regime,
            "trials": cell_results,
            "elapsed_sec": round(time.time() - cell_start, 1),
        }
        results["cells"][cell_key] = cell_data

        cell_idx += 1
        elapsed_total = time.time() - sweep_start
        cells_done_this_run = sum(1 for k in results["cells"] if k not in getattr(run_single_trial, '_preloaded', set()))
        est_remaining = (elapsed_total / max(cell_idx - completed, 1)) * (total_cells - cell_idx)
        print(f"  [{cell_idx}/{total_cells}] keys={nk:>2}, updates={nu:>3}: "
              f"RI={ri_acc:.0%} [{cell_stats['RI']['ci_lower']:.0%}-{cell_stats['RI']['ci_upper']:.0%}] "
              f"PI={pi_acc:.0%} [{cell_stats['PI']['ci_lower']:.0%}-{cell_stats['PI']['ci_upper']:.0%}] "
              f"regime={regime} ({cell_data['elapsed_sec']:.1f}s) "
              f"[ETA: {est_remaining/60:.0f}min]")

        # Save every 5 cells
        if cell_idx % 5 == 0:
            save_results(results, partial=True)

results["end_time"] = time.strftime("%Y-%m-%d %H:%M:%S")
save_results(results, partial=False)
print(f"\nSweep complete! {len(results['cells'])} cells in {(time.time()-sweep_start)/60:.1f} minutes")

  [16/175] keys=2, updates=180: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [17/175] keys=2, updates=200: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [61/175] keys=7, updates=60: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [62/175] keys=7, updates=80: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [63/175] keys=7, updates=100: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [64/175] keys=7, updates=120: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [65/175] keys=7, updates=140: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [66/175] keys=7, updates=160: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [67/175] keys=7, updates=180: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [68/175] keys=7, updates=200: SKIPPED (saturated - RI zeros=0, PI zeros=3)
  [81/175] keys=10, updates=120: RI=0% [0%-0%] PI=0% [0%-0%] regime=C (50.4s) [ETA: 7min]
  [82/175] keys=10, updates=140: RI=0% [0%-0%] PI=0% [0%-0%] regime=C (24.2s) [ETA: 10min]
  [83/175] keys=10, updates=160: RI=0% [0%-0%] PI=0

## Results Summary

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Visualization: Heatmaps
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for ax_idx, (title, data_fn) in enumerate([
    ("RI Accuracy", lambda c: c["stats"]["RI"]["accuracy"]),
    ("PI Accuracy", lambda c: c["stats"]["PI"]["accuracy"]),
    ("RI - PI (Asymmetry)", lambda c: c["stats"]["RI"]["accuracy"] - c["stats"]["PI"]["accuracy"]),
]):
    ax = axes[ax_idx]
    grid = np.full((len(KEY_LEVELS), len(UPDATE_LEVELS)), np.nan)
    for i, nk in enumerate(KEY_LEVELS):
        for j, nu in enumerate(UPDATE_LEVELS):
            cell_key = f"{nk}_{nu}"
            if cell_key in results["cells"]:
                grid[i, j] = data_fn(results["cells"][cell_key])
    
    if "Asymmetry" in title:
        cmap = "RdBu_r"
        vmin, vmax = -1, 1
    else:
        cmap = "RdYlGn"
        vmin, vmax = 0, 1
    
    im = ax.imshow(grid, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(UPDATE_LEVELS)))
    ax.set_xticklabels(UPDATE_LEVELS, rotation=45, fontsize=7)
    ax.set_yticks(range(len(KEY_LEVELS)))
    ax.set_yticklabels(KEY_LEVELS)
    ax.set_xlabel("Updates per key")
    ax.set_ylabel("Number of keys")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f"Behavioral Landscape: {MODEL_NAME}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/behavioral_landscape_{MODEL_NAME.split('/')[-1]}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved heatmap to {FIGURES_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Copy results to notebook output (for VSCode-connected Colab)
# ═══════════════════════════════════════════════════════════════════════════
import glob

print("=== Files saved ===")
for f in sorted(glob.glob(f"{SAVE_DIR}/*.json") + glob.glob(f"{FIGURES_DIR}/*.png")):
    size = os.path.getsize(f)
    print(f"  {f} ({size:,} bytes)")

# Print summary JSON (small file) so it appears in notebook output
# VSCode will sync the notebook with this output to your local machine
model_short = MODEL_NAME.split("/")[-1]
summary_path = f"{SAVE_DIR}/behavioral_sweep_{model_short}.json"
if os.path.exists(summary_path):
    print(f"\n=== Summary (copy this if needed) ===")
    with open(summary_path) as f:
        print(f.read())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Visualization: Decay Curves
# ═══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(3, 4, figsize=(20, 12), sharex=False, sharey=True)
axes = axes.flatten()

for idx, nk in enumerate(KEY_LEVELS):
    ax = axes[idx]
    ri_accs, ri_lo, ri_hi = [], [], []
    pi_accs, pi_lo, pi_hi = [], [], []
    valid_updates = []
    
    for nu in UPDATE_LEVELS:
        cell_key = f"{nk}_{nu}"
        if cell_key in results["cells"]:
            c = results["cells"][cell_key]
            valid_updates.append(nu)
            ri_accs.append(c["stats"]["RI"]["accuracy"])
            ri_lo.append(c["stats"]["RI"]["ci_lower"])
            ri_hi.append(c["stats"]["RI"]["ci_upper"])
            pi_accs.append(c["stats"]["PI"]["accuracy"])
            pi_lo.append(c["stats"]["PI"]["ci_lower"])
            pi_hi.append(c["stats"]["PI"]["ci_upper"])
    
    if valid_updates:
        ax.plot(valid_updates, ri_accs, 'b-o', markersize=3, label='RI', linewidth=1.5)
        ax.fill_between(valid_updates, ri_lo, ri_hi, alpha=0.2, color='blue')
        ax.plot(valid_updates, pi_accs, 'r-s', markersize=3, label='PI', linewidth=1.5)
        ax.fill_between(valid_updates, pi_lo, pi_hi, alpha=0.2, color='red')
    
    ax.set_title(f"Keys={nk}", fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_xscale('log')
    if idx >= 8:
        ax.set_xlabel("Updates")
    if idx % 4 == 0:
        ax.set_ylabel("Accuracy")
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle(f"RI vs PI Decay Curves: {MODEL_NAME}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/decay_curves_{MODEL_NAME.split('/')[-1]}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved decay curves to {FIGURES_DIR}")